In [15]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [16]:
import ollama
response = ollama.chat(model="llama3.1:8b",
                       messages=[{"role": "user", "content": "Say hello in one sentence."}])
print(response["message"]["content"])

Hello! Is there something I can help you with today?


In [17]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("lufthansa")     # reopen the existing store

print("Loaded collection with", collection.count(), "docs")

Loaded collection with 195 docs


In [18]:
question = "What are the biggest risks for Lufthansa right now?"

# retrieve the 5 most relevant docs (semantic search)
results   = collection.query(query_texts=[question], n_results=5)
retrieved = results["documents"][0]
metas     = results["metadatas"][0]

# build ONE context string from the retrieved docs (tag each with its source)
context = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

print(context)

[news] Lufthansa at a disadvantage compared to competitors. . Lufthansa maintained its financial forecasts for 2025 and said on Tuesday it was optimistic for the summer season, adopting a more upbeat tone .

[news] Lufthansa sticks to 2026 outlook despite $2 billion jet-fuel hit, shares . May 6, 2026 . 2026 FORECAST TO STAY, IF NO FURTHER STRIKES OCCUR. It maintained its forecast for 2026 of a significantly higher adjusted operating .

[news] Current information | Lufthansa. 3 weeks ago - Get the latest flight information including updates on routes, cancellations due to weather conditions or strikes, rebooking options, and more.

[news] Current information | Lufthansa. Get the latest flight information including updates on routes, cancellations due to weather conditions or strikes, rebooking options, and more.

[news] Lufthansa to cut 4,000 jobs, raises profitability targets -. We definitely lag behind some of our competitors when it comes to financial performance," Chief Executive Ca

testing ollama with correct system_prompt and user_prompt

In [19]:
import ollama

system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided to answer — do not invent facts.
Be concise, specific, and ground every claim in the evidence."""

user_prompt = f"""Evidence:
{context}

Question: {question}

Answer as a strategic advisor, citing the evidence."""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
)

print(response["message"]["content"])

Based on the provided evidence, I identify the following significant risks for Lufthansa:

1. **Strikes**: The article from May 6, 2026, explicitly states that if no further strikes occur, the company will maintain its forecast for 2026. This suggests that ongoing strike activities pose a considerable risk to Lufthansa's financial performance.
2. **Weather conditions**: Another article mentions cancellations due to weather conditions as part of the current information provided by Lufthansa. This indicates that adverse weather can impact flight operations and potentially disrupt revenue streams.

These two factors are highlighted as significant risks, as they have a direct impact on Lufthansa's ability to meet its financial forecasts and maintain competitiveness in the market.


In [20]:
import json, ollama

system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "supporting_evidence": list of 2-3 short evidence points taken from the context
- "expected_impact": the expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""

user_prompt = f"""Evidence:
{context}

Question: {question}"""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    format="json"          
)

rec = json.loads(response["message"]["content"])   
print(json.dumps(rec, indent=2))                   

{
  "recommendation": "Invest in cost-cutting measures and efficiency improvements",
  "supporting_evidence": [
    "Lufthansa maintained its financial forecasts for 2025, but this may be overly optimistic considering it lags behind competitors.",
    "The company is expecting a $2 billion jet-fuel hit, which could have significant implications for future profitability.",
    "Lufthansa plans to cut 4,000 jobs as part of cost-cutting efforts, highlighting the need for further measures."
  ],
  "expected_impact": "Improved competitiveness and reduced financial risks",
  "risk_level": "High",
  "priority": "High"
}


reusable ceo_agent(question) function for my dashboard

In [21]:
def ceo_agent(question, k=5):
    # 1. RETRIEVE evidence
    results   = collection.query(query_texts=[question], n_results=k)
    retrieved = results["documents"][0]
    metas     = results["metadatas"][0]
    context   = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

    # 2. PROMPT
    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    user_prompt = f"Evidence:\n{context}\n\nQuestion: {question}"

    # 3. GENERATE (structured JSON)
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )

    # 4. PARSE + attach question and evidence URLs
    rec = json.loads(response["message"]["content"])
    rec["question"] = question
    rec["sources"]  = [m["url"] for m in metas]
    return rec

testing reusable code

In [22]:
import json
result = ceo_agent("What are the major opportunities for Lufthansa?")
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "recommendation": "Invest in cost reduction initiatives through restructuring",
  "justification": "Given Lufthansa's underperformance compared to competitors, as stated by CEO Carsten Spohr, focusing on cost reduction is crucial. Additionally, the company plans to retire a large portion of its long-haul fleet and shut down operations, indicating a willingness to restructure.",
  "supporting_evidence": [
    "Lufthansa Review states that Lufthansa is a mid-range performing brand within its category",
    "CEO Carsten Spohr's statement on financial underperformance",
    "Plan to retire large portion of long-haul fleet"
  ],
  "expected_impact": "Improved financial performance and competitiveness",
  "risk_level": "Medium",
  "priority": "High",
  "question": "What are the major opportunities for Lufthansa?",
  "sources": [
    "https://report.lufthansagroup.com/2024/annual-report/en/downloads/",
    "https://lufthansa.knoji.com/",
    "https://www.myjoyonline.com/lufthansa-to-cut-4

In [23]:
questions = [
    "What are the major opportunities for Lufthansa?",
    "What are the biggest risks for Lufthansa?",
    "What are competitors doing?",
    "Which technologies or trends should Lufthansa management monitor?",
    "What strategic actions should Lufthansa prioritize?",
]

recommendations = []
for q in questions:
    print("Generating:", q)
    recommendations.append(ceo_agent(q))

json.dump(recommendations,
          open("recommendations.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n Saved", len(recommendations), "recommendations")

Generating: What are the major opportunities for Lufthansa?
Generating: What are the biggest risks for Lufthansa?
Generating: What are competitors doing?
Generating: Which technologies or trends should Lufthansa management monitor?
Generating: What strategic actions should Lufthansa prioritize?

 Saved 5 recommendations


CEO Briefing (Section 7)

In [24]:
def ceo_briefing(recommendations):
    # summarize the recommendations as input
    rec_summary = "\n".join(
        f"- {r['recommendation']} (priority {r['priority']}, risk {r['risk_level']})"
        for r in recommendations
    )

    system_prompt = """You are chief of staff to the CEO of Lufthansa.
Write a concise executive briefing as a JSON object with EXACTLY these 3 keys:
- "what_happened": a single plain-text string (2-3 sentences)
- "why_it_matters": a single plain-text string (2-3 sentences)
- "what_to_do_next": a single plain-text string (2-3 sentences)
Each value MUST be a plain string — NOT a nested object, dict, or list.
Base it ONLY on the recommendations provided. Do not invent facts."""

    user_prompt = f"Strategic recommendations:\n{rec_summary}\n\nWrite the CEO briefing."

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )
    return json.loads(response["message"]["content"])

In [25]:
briefing = ceo_briefing(recommendations)
json.dump(briefing, open("ceo_briefing.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(json.dumps(briefing, indent=2, ensure_ascii=False))

{
  "what_happened": "Lufthansa has identified key areas of focus to maintain its competitive edge: retaining core competencies in mid-range performance category, investing in cost reduction initiatives, implementing competitor analysis, and accelerating fleet renewal.",
  "why_it_matters": "These strategic recommendations are crucial to ensuring Lufthansa's continued success in the face of intensifying industry competition. By prioritizing these areas, we can preserve our market position and drive growth.",
  "what_to_do_next": "We must allocate sufficient resources to execute these initiatives effectively, starting with a thorough cost reduction analysis and implementation plan, followed by a comprehensive competitor analysis program, and accelerated fleet renewal efforts."
}
